# Poznań Rent Price Prediction

End-to-end machine learning pipeline for predicting apartment rental prices from incomplete tabular and textual real-estate listing data.

## 1 Introduction

This notebook presents a machine learning solution for predicting apartment rental prices from real-estate listing data. The task is a supervised regression problem where the target variable is `price`, and the input data contains both structured tabular attributes and short textual information from the listing title (`ad_title`).

The solution focuses on a reproducible end-to-end modeling pipeline: data loading, missing-value normalization, leakage-safe feature engineering, cross-validation, base model training, stacking, and final submission export. The final approach combines several complementary models: tree-based regressors for tabular features, a TF-IDF + Ridge model for title text, and a Polynomial Ridge meta-model that learns how to combine the base predictions.


## 2 Methodology

The notebook implements a complete machine learning pipeline, from loading the data to generating the final submission file. First, missing values encoded as sentinel values (for example `-999`) are converted to `NaN`, so that downstream imputation works consistently. The target vector is defined as `y = train['price']`, and reproducibility is controlled through a fixed `RANDOM_STATE` and `KFold` cross-validation with `shuffle=True`. The notebook also checks that the training and test feature sets are aligned, which reduces the risk of errors during prediction and submission generation.

Feature engineering is performed without using the target variable, which avoids leakage. Date columns are transformed into features describing the listing lifecycle, such as `listing_duration` and `days_to_modif`, together with seasonal features like `activ_year`, `activ_month`, and `activ_dow`. Suspicious date-derived values are flagged and then replaced with `NaN`, preserving anomaly information through binary indicators. Listing titles are normalized into `ad_title_filled`, and simple text meta-features are added, such as title length, word count, and digit presence. The `quarter` categorical variable is cleaned into `quarter_clean`, and rare categories are mapped to `other`, which stabilizes one-hot encoding and prevents unnecessary feature-space growth. Missingness indicators (`*_missing`) are also added for important numeric variables, because the fact that a value is missing can itself be predictive in listing data.

For tabular modeling, the solution uses LightGBM and XGBoost. The tabular preprocessing is built with `scikit-learn` using a `ColumnTransformer`: numeric columns are imputed with the median, while categorical columns are imputed with the most frequent value and then one-hot encoded with `handle_unknown="ignore"`. The improved tabular variant also treats some zero values as missing values for selected variables, such as rent and deposit, and extracts additional hints from listing titles, such as area and number of rooms.

For text modeling, listing titles are represented with a combination of word-level and character-level TF-IDF features. A Ridge regression model is trained on those features, and the regularization strength `alpha` is selected using out-of-fold (OOF) validation. A hybrid text-tabular model is also added: TF-IDF features are compressed with `TruncatedSVD`, and the resulting dense components are appended to the tabular feature set for a LightGBM model. This gives the tree-based model access to textual signal in a compact numeric form.

The final prediction is produced with stacking. The meta-feature matrix `X_meta` is built from OOF predictions of four base models: improved LightGBM tabular, XGBoost, LightGBM with SVD text components, and the TF-IDF + Ridge text model. The test meta-feature matrix is constructed analogously. A `PolynomialFeatures(degree=2) + StandardScaler + Ridge` pipeline is then trained as the meta-model. Polynomial interactions allow the ensemble to learn not only linear weights for the base models, but also interactions between them. The final predictions are clipped to the training target range and saved as a Kaggle-compatible submission file.


## 3 Results

Model quality was evaluated locally with 5-fold cross-validation using mean squared error (MSE) on out-of-fold (OOF) predictions. The final submission was also evaluated externally on Kaggle. Lower MSE values indicate better predictive performance.

| Model | Main feature set | Validation setup | OOF MSE |
|---|---|---:|---:|
| LightGBM tabular | engineered tabular features | 5-fold CV | 90,326 |
| XGBoost tabular | engineered tabular features | 5-fold CV | 89,817 |
| TF-IDF + Ridge | listing title text | 5-fold CV | 155,122 |
| LightGBM + SVD text | tabular features + compressed title text | 5-fold CV | 82,691 |
| Polynomial Ridge stacker | OOF predictions from all base models | 5-fold CV | **75,757** |

The results improved as the solution moved from individual models to an ensemble. The tabular models captured structured relationships between listing attributes and price, while the text model added complementary information from title wording, abbreviations, and numeric hints. The LightGBM + SVD model gave the strongest individual base-model result, showing that compressed title text improved the tree-based representation.

The final model uses stacking with a `PolynomialFeatures(degree=2) + StandardScaler + Ridge` meta-model. This setup allows the final estimator to learn both direct weights for each base model and second-order interactions between their predictions. The best local stacking configuration achieved an OOF MSE of approximately **75,757**, and the final Kaggle score was **71,243.37**.


## 4 Summary

The notebook implements a complete and reproducible rental-price prediction pipeline based on both tabular and text features. The strongest results came from combining heterogeneous models rather than relying on a single estimator. Tree-based models handled nonlinear relationships in structured variables, while the TF-IDF + Ridge model contributed complementary information from listing titles.

A key methodological component is the use of OOF predictions for stacking. Each meta-feature is generated by a model that did not train on the corresponding row, which reduces leakage and makes local validation more reliable. The final Polynomial Ridge stacker is simple, regularized, and effective for combining several correlated base predictions.

Missing-value handling was also important. Sentinel values were first converted to `NaN`, then processed through consistent imputation pipelines. Additional missingness and anomaly indicators preserved information about originally absent or suspicious values. This combination improved robustness while keeping the feature engineering leakage-safe.

The final notebook keeps intermediate outputs visible for review on GitHub, uses repository-friendly relative paths, and exports the final prediction file to the `outputs/` directory.


### Imports and reproducibility

This block initializes the environment used by the full pipeline: cross-validation, OOF prediction generation, stacking, and submission creation. It imports tools for tabular preprocessing (imputation and one-hot encoding), text modeling (TF-IDF and `FeatureUnion`), dimensionality reduction (`TruncatedSVD`), base models (LightGBM and XGBoost), and the final meta-model (`PolynomialFeatures` + Ridge).

Setting `RANDOM_STATE` and `np.random.seed` makes KFold splits and model results more reproducible. The `pandas` display options are only used for easier inspection and do not affect model training.


In [ ]:
import re
from pathlib import Path

import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

from lightgbm import LGBMRegressor
from lightgbm.callback import early_stopping

from xgboost import XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Keep notebook outputs focused on model results rather than library warnings.
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Environment ready. Random state:", RANDOM_STATE)


Environment ready. Random state: 42


### Loading data and preparing the target

This step loads `train`, `test`, and `sample_sub` from repository-friendly relative paths and defines `TARGET = "price"`. The two assertions protect the modeling pipeline from leakage: the target must exist in the training data and must not exist in the test data. The vector `y = train[TARGET]` is then used in cross-validation, OOF prediction generation, and final stacking.

The printed shapes are a quick sanity check confirming that the loaded files have the expected dimensions and that `sample_sub` matches the test set size.


### Path configuration

The notebook assumes the following GitHub-friendly repository structure:

```text
.
├── data/
│   ├── pzn-rent-train.csv
│   ├── pzn-rent-test.csv
│   └── pzn-sample-sub.csv
├── notebooks/
│   └── poznan_rent_price_prediction.ipynb
└── outputs/
    └── submission_polystack.csv
```

The path logic below works both when the notebook is launched from the repository root and when it is launched from the `notebooks/` directory. The `PROJECT_ROOT` environment variable can also be used to override the root path if needed.


In [ ]:
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "pzn-rent-train.csv"
test_path = DATA_DIR / "pzn-rent-test.csv"
sub_path = DATA_DIR / "pzn-sample-sub.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sub_path)

TARGET = "price"
assert TARGET in train.columns, "Target 'price' not found in train!"
assert TARGET not in test.columns, "Target 'price' should not be in test!"

y = train[TARGET].copy()

print("Loaded:")
print("  train:", train.shape)
print("  test :", test.shape)
print("  sample_sub:", sample_sub.shape)


Loaded:
  train: (11297, 36)
  test : (4842, 35)
  sample_sub: (4842, 2)


### Missing-value normalization: sentinel values to `NaN`

Some missing values are encoded as special sentinel values, such as `-999`. This step converts such values to `NaN`, so that later preprocessing stages handle them correctly. In particular, `SimpleImputer` can then treat them as missing values instead of real numeric observations. This improves the stability of numeric feature distributions and makes imputation and one-hot encoding more consistent across the LightGBM, XGBoost, and hybrid text-tabular pipelines.


In [ ]:
def normalize_missing_values(df: pd.DataFrame, sentinel_values=(-999,)) -> pd.DataFrame:
    df = df.copy()
    for sentinel in sentinel_values:
        mask = (df == sentinel)
        cols = mask.any()[mask.any()].index.tolist()
        for col in cols:
            df.loc[mask[col], col] = np.nan
    return df

train = normalize_missing_values(train, sentinel_values=(-999,))
test  = normalize_missing_values(test, sentinel_values=(-999,))

print("Sentinels normalized.")


Sentinels normalized.


### Feature engineering without target leakage

This block creates the main engineered features used by the tabular and hybrid models. All transformations are based only on input variables, without using the target, which prevents leakage.

The function starts by normalizing title text into `ad_title_filled`, then creates simple text meta-features: character length, word count, and a flag indicating whether the title contains digits. Date columns are converted with `pd.to_datetime`, and listing-lifecycle features are derived from them, including `listing_duration`, `days_to_modif`, activation year, activation month, and day of week. Suspicious date-derived values are flagged and replaced with `NaN`, so that the model can use both the cleaned value and the information that the original value was anomalous.

The `quarter` variable is cleaned into `quarter_clean`: missing or placeholder values are mapped to `unknown`, while rare categories are grouped into `other`. This makes one-hot encoding more stable. The block also adds missingness indicators for selected numeric features, such as area, number of rooms, rent, deposit, and date-derived features. These indicators are useful because missing information in real-estate listings can itself carry predictive signal.

Finally, the same feature engineering is applied to both `train` and `test`, and a check confirms that all test columns are present in the training feature table, except for the target column.


In [ ]:
def add_date_features(df: pd.DataFrame, date_cols=("date_activ", "date_modif", "date_expire")) -> pd.DataFrame:
    df = df.copy()
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if all(c in df.columns for c in date_cols):
        duration = (df["date_expire"] - df["date_activ"]).dt.days
        to_modif = (df["date_modif"] - df["date_activ"]).dt.days

        df["listing_duration"] = duration
        df["days_to_modif"] = to_modif

        df["activ_year"] = df["date_activ"].dt.year
        df["activ_month"] = df["date_activ"].dt.month
        df["activ_dow"] = df["date_activ"].dt.dayofweek

        df["duration_nonpositive"] = (duration <= 0).astype(int)
        df["duration_ge_365"] = (duration >= 365).astype(int)
        df["to_modif_nonpositive"] = (to_modif <= 0).astype(int)
        df["to_modif_ge_365"] = (to_modif >= 365).astype(int)

        df.loc[(duration <= 0) | (duration >= 365), "listing_duration"] = np.nan
        df.loc[(to_modif <= 0) | (to_modif >= 365), "days_to_modif"] = np.nan

        df.drop(columns=list(date_cols), inplace=True, errors="ignore")

    return df

def add_text_and_category_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_df = train_df.copy()
    test_df  = test_df.copy()

    TEXT_RAW = "ad_title"
    TEXT_COL = "ad_title_filled"

    for df in (train_df, test_df):
        df[TEXT_COL] = df[TEXT_RAW].fillna("").astype(str).str.lower() if TEXT_RAW in df.columns else ""
        df["ad_title_len"] = df[TEXT_COL].str.len()
        df["ad_title_words"] = df[TEXT_COL].str.split().str.len()
        df["ad_title_has_digit"] = df[TEXT_COL].str.contains(r"\d").astype(int)

    if "quarter" in train_df.columns:
        for df in (train_df, test_df):
            df["quarter_missing"] = df["quarter"].isna().astype(int)
            q = df["quarter"].astype(str).str.lower().str.strip().replace("nan", np.nan)
            df["quarter"] = q

        counts = train_df["quarter"].value_counts(dropna=True)
        rare = counts[counts < 20].index

        for df in (train_df, test_df):
            df["quarter_clean"] = df["quarter"].where(~df["quarter"].isin(rare), other="other")
            df["quarter_clean"] = df["quarter_clean"].fillna("unknown")

    return train_df, test_df

def add_missing_markers_and_outliers(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_df = train_df.copy()
    test_df  = test_df.copy()

    marker_cols = ["flat_area", "flat_rooms", "flat_rent", "flat_deposit", "building_floor_num",
                   "listing_duration", "days_to_modif"]
    marker_cols = [c for c in marker_cols if c in train_df.columns]

    for df in (train_df, test_df):
        for col in marker_cols:
            df[f"{col}_missing"] = df[col].isna().astype(int)

        if "flat_rent" in df.columns:
            df["flat_rent_is_zero"] = (df["flat_rent"] == 0).astype(int)
        if "flat_deposit" in df.columns:
            df["flat_deposit_is_zero"] = (df["flat_deposit"] == 0).astype(int)

    if "flat_area" in train_df.columns:
        area_q_hi = train_df["flat_area"].quantile(0.995)
        area_upper = float(min(250, area_q_hi)) if pd.notna(area_q_hi) else 250.0
        area_lower = 8.0
        for df in (train_df, test_df):
            area = df["flat_area"]
            df["flat_area_too_small"] = (area < area_lower).astype(int)
            df["flat_area_too_big"]   = (area > area_upper).astype(int)
            df.loc[(area < area_lower) | (area > area_upper), "flat_area"] = np.nan

    if "flat_rooms" in train_df.columns:
        rooms_lower, rooms_upper = 1.0, 10.0
        for df in (train_df, test_df):
            rooms = df["flat_rooms"]
            df["flat_rooms_too_small"] = (rooms < rooms_lower).astype(int)
            df["flat_rooms_too_big"]   = (rooms > rooms_upper).astype(int)
            df.loc[(rooms < rooms_lower) | (rooms > rooms_upper), "flat_rooms"] = np.nan

    return train_df, test_df

train_fe = add_date_features(train)
test_fe  = add_date_features(test)

train_fe, test_fe = add_text_and_category_features(train_fe, test_fe)
train_fe, test_fe = add_missing_markers_and_outliers(train_fe, test_fe)

print("FE done:")
print("  train_fe:", train_fe.shape)
print("  test_fe :", test_fe.shape)


FE done:
  train_fe: (11297, 61)
  test_fe : (4842, 60)


### Preparing tabular features

This step defines the feature matrix used by the tabular models. Columns that should not be used as direct predictors are removed: the target (`price`), the row identifier (`id`), and the raw `quarter` column, because its cleaned version `quarter_clean` is used instead. The notebook then verifies that the training and test matrices have exactly the same feature columns.

Next, columns are split into numeric and categorical groups. Numeric columns are passed through median imputation, while categorical columns are imputed with the most frequent value and one-hot encoded with `handle_unknown="ignore"`. This setup is important for reproducibility and for safe prediction on the test set, because categories present only in the test data will not break the pipeline.


In [ ]:
TEXT_COL = "ad_title_filled"

area_pattern  = re.compile(r"(\d+(?:[.,]\d+)?)\s*(m2|m²)")
rooms_pattern = re.compile(r"(\d+)\s*(-?\s*pok)")

def extract_area(text: str):
    if not isinstance(text, str) or not text:
        return np.nan
    m = area_pattern.search(text)
    if not m:
        return np.nan
    return float(m.group(1).replace(",", "."))

def extract_rooms(text: str):
    if not isinstance(text, str) or not text:
        return np.nan
    m = rooms_pattern.search(text)
    if not m:
        return np.nan
    return float(m.group(1))

def to_bool01_series(s: pd.Series) -> pd.Series:
    mapping = {True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0}
    return s.map(mapping)

def build_improved_tabular(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = train_df.copy()
    te = test_df.copy()

    DROP_COLS = ["price", "id", "quarter", "ad_title"]
    tr.drop(columns=DROP_COLS, errors="ignore", inplace=True)
    te.drop(columns=DROP_COLS, errors="ignore", inplace=True)

    for df in (tr, te):
        df["title_area"]  = df[TEXT_COL].apply(extract_area) if TEXT_COL in df.columns else np.nan
        df["title_rooms"] = df[TEXT_COL].apply(extract_rooms) if TEXT_COL in df.columns else np.nan

    AREA_LO, AREA_HI = 8.0, 250.0
    ROOMS_LO, ROOMS_HI = 1.0, 10.0

    for df in (tr, te):
        df["title_area_ok"]  = ((df["title_area"]  >= AREA_LO)  & (df["title_area"]  <= AREA_HI)).astype(int)
        df["title_rooms_ok"] = ((df["title_rooms"] >= ROOMS_LO) & (df["title_rooms"] <= ROOMS_HI)).astype(int)

        if "flat_area" in df.columns:
            m = df["flat_area"].isna() & (df["title_area_ok"] == 1)
            df["flat_area_filled_from_title"] = m.astype(int)
            df.loc[m, "flat_area"] = df.loc[m, "title_area"]
        else:
            df["flat_area_filled_from_title"] = 0

        if "flat_rooms" in df.columns:
            m = df["flat_rooms"].isna() & (df["title_rooms_ok"] == 1)
            df["flat_rooms_filled_from_title"] = m.astype(int)
            df.loc[m, "flat_rooms"] = df.loc[m, "title_rooms"]
        else:
            df["flat_rooms_filled_from_title"] = 0

        if "flat_rent" in df.columns:
            df["flat_rent_was_zero"] = (df["flat_rent"] == 0).astype(int)
            df["flat_rent_clean"] = df["flat_rent"].replace(0, np.nan)
        if "flat_deposit" in df.columns:
            df["flat_deposit_was_zero"] = (df["flat_deposit"] == 0).astype(int)
            df["flat_deposit_clean"] = df["flat_deposit"].replace(0, np.nan)

    for col in tr.columns:
        if tr[col].dtype == "object" and col not in [TEXT_COL, "quarter_clean"]:
            uniq = set(tr[col].dropna().unique().tolist())
            if uniq.issubset({True, False, "True", "False", "true", "false"}):
                tr[col] = to_bool01_series(tr[col]).astype("float")
                if col in te.columns:
                    te[col] = to_bool01_series(te[col]).astype("float")

    for df in (tr, te):
        if "flat_area" in df.columns and "flat_rooms" in df.columns:
            df["area_per_room"] = df["flat_area"] / df["flat_rooms"]
            df.loc[~np.isfinite(df["area_per_room"]), "area_per_room"] = np.nan

    return tr, te

train2, test2 = build_improved_tabular(train_fe, test_fe)

X_tab = train2.drop(columns=[TEXT_COL], errors="ignore")
X_tab_test = test2.drop(columns=[TEXT_COL], errors="ignore")
y_target = y.values

missing_in_test = sorted(set(X_tab.columns) - set(X_tab_test.columns))
extra_in_test   = sorted(set(X_tab_test.columns) - set(X_tab.columns))
assert not missing_in_test and not extra_in_test, f"Train/test mismatch: missing={missing_in_test}, extra={extra_in_test}"

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Tabular matrices ready:", X_tab.shape, X_tab_test.shape)


Tabular matrices ready: (11297, 67) (4842, 67)


### Cross-validation setup

This block defines the target array and the shared 5-fold cross-validation strategy used across the base models and the meta-model. Using the same `KFold` configuration makes model comparisons easier and keeps the OOF predictions aligned row by row.

The `assert` statement checks that the target vector contains no missing values. This is a basic but important safety check before fitting regression models.


In [ ]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])
    cat_tf = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_tf, numeric_cols),
            ("cat", cat_tf, categorical_cols),
        ],
        sparse_threshold=1.0
    )

print("Ready.")


Ready.


### Base model #1: LightGBM on enhanced tabular features

This function trains the LightGBM tabular model and returns two outputs: OOF predictions for the training set and predictions for the test set. The OOF predictions are later used as meta-features in the final stacking model.

For each fold, the preprocessing pipeline is fitted only on the training split and then applied to the validation split. This prevents validation leakage. LightGBM is trained with early stopping on the validation fold, and the best iteration is stored for each fold. After cross-validation, the final model is trained on the full training set using the median best iteration, which is a stable compromise across folds.

The function also prints fold-level MSE values and the overall OOF MSE, making it easy to compare this model with other base models.


In [ ]:
def fit_lgb_tabular_oof(X: pd.DataFrame, X_test: pd.DataFrame, y: np.ndarray):
    pre = build_preprocessor(X)
    oof = np.zeros(len(X))
    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        X_tr_t = pre.fit_transform(X_tr)
        X_val_t = pre.transform(X_val)

        lgbm = LGBMRegressor(
            n_estimators=20000,
            learning_rate=0.02,
            num_leaves=127,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1
        )
        lgbm.fit(
            X_tr_t, y_tr,
            eval_set=[(X_val_t, y_val)],
            eval_metric="l2",
            callbacks=[early_stopping(stopping_rounds=300, verbose=False)]
        )

        pred = lgbm.predict(X_val_t, num_iteration=lgbm.best_iteration_)
        oof[val_idx] = pred
        best_iters.append(lgbm.best_iteration_)
        print(f"Fold {fold+1} MSE: {mean_squared_error(y_val, pred):.4f} | best_iter={lgbm.best_iteration_}")

    best_n = int(np.median(best_iters))
    print("\nLGB Tabular CV MSE:", mean_squared_error(y, oof))
    print("Best iters (median):", best_n)

    X_full_t = pre.fit_transform(X)
    X_test_t = pre.transform(X_test)

    lgbm_final = LGBMRegressor(
        n_estimators=best_n,
        learning_rate=0.02,
        num_leaves=127,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    lgbm_final.fit(X_full_t, y)
    test_pred = lgbm_final.predict(X_test_t)

    return oof, test_pred

oof_pred_tab2, test_pred_tab2 = fit_lgb_tabular_oof(X_tab, X_tab_test, y_target)


Fold 1 MSE: 84219.5007 | best_iter=876
Fold 2 MSE: 91419.9679 | best_iter=790
Fold 3 MSE: 92187.4446 | best_iter=784
Fold 4 MSE: 90390.0722 | best_iter=673
Fold 5 MSE: 93415.5466 | best_iter=1033

LGB Tabular CV MSE: 90326.06260519834
Best iters (median): 790


### Base model #2: XGBoost on the same tabular feature set

This block trains an XGBoost regressor on the same tabular features and preprocessing as the LightGBM model. The purpose is not only to improve individual performance, but also to increase diversity inside the ensemble. Even when XGBoost and LightGBM use the same features, their different tree-building strategies can lead to different errors, which is useful for stacking.

As before, preprocessing is fitted inside each fold, OOF predictions are generated for the training set, and a final model is trained on all training data to produce test predictions. The printed fold scores and overall OOF MSE are used to assess both quality and stability.


In [ ]:
def fit_xgb_oof(X: pd.DataFrame, X_test: pd.DataFrame, y: np.ndarray):
    pre = build_preprocessor(X)
    oof = np.zeros(len(X))

    XGB_PARAMS = dict(
        n_estimators=4000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        gamma=0.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        X_tr_t = pre.fit_transform(X_tr)
        X_val_t = pre.transform(X_val)

        xgb = XGBRegressor(**XGB_PARAMS)
        xgb.fit(X_tr_t, y_tr)

        pred = xgb.predict(X_val_t)
        oof[val_idx] = pred
        print(f"Fold {fold+1} MSE: {mean_squared_error(y_val, pred):.4f}")

    print("\nXGBoost CV MSE:", mean_squared_error(y, oof))

    X_full_t = pre.fit_transform(X)
    X_test_t = pre.transform(X_test)

    xgb_final = XGBRegressor(**XGB_PARAMS)
    xgb_final.fit(X_full_t, y)
    test_pred = xgb_final.predict(X_test_t)

    return oof, test_pred

oof_pred_xgb, test_pred_xgb = fit_xgb_oof(X_tab, X_tab_test, y_target)


Fold 1 MSE: 85174.3438
Fold 2 MSE: 92751.5703
Fold 3 MSE: 88076.8594
Fold 4 MSE: 90084.5078
Fold 5 MSE: 92996.3594

XGBoost CV MSE: 89816.57655873697


### Base model #3: TF-IDF + Ridge text model

This block builds a pure text model based only on listing titles. The input is `ad_title_filled`, which contains normalized title text. The model uses a `FeatureUnion` of word-level TF-IDF features and character-level TF-IDF features. Word n-grams capture meaningful phrases, while character n-grams are useful for spelling variants, abbreviations, and inflected forms.

Ridge regression is then trained on the sparse TF-IDF matrix. Several `alpha` values are tested, and the best one is selected based on OOF MSE. The OOF predictions from this model are important for stacking, because they represent information that is not fully captured by tabular models.

After selecting the best `alpha`, the model is trained on the full training text and used to generate predictions for the test set.


In [ ]:
X_text = train_fe[TEXT_COL].fillna("").astype(str).values
X_text_test = test_fe[TEXT_COL].fillna("").astype(str).values

def fit_text2_oof(train_text: np.ndarray, test_text: np.ndarray, y: np.ndarray):
    alphas = [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
    best = {"alpha": None, "mse": np.inf}
    best_oof, best_test = None, None

    for alpha in alphas:
        model = Pipeline([
            ("features", FeatureUnion([
                ("word", TfidfVectorizer(
                    analyzer="word", ngram_range=(1, 2),
                    min_df=3, max_df=0.9,
                    strip_accents="unicode", lowercase=True, sublinear_tf=True
                )),
                ("char", TfidfVectorizer(
                    analyzer="char", ngram_range=(3, 5),
                    min_df=3, max_df=0.95,
                    strip_accents="unicode", lowercase=True, sublinear_tf=True
                )),
            ])),
            ("ridge", Ridge(alpha=alpha, random_state=RANDOM_STATE))
        ])

        oof = np.zeros(len(train_text))
        for tr_idx, val_idx in kf.split(train_text):
            model.fit(train_text[tr_idx], y[tr_idx])
            oof[val_idx] = model.predict(train_text[val_idx])

        mse = mean_squared_error(y, oof)
        print(f"alpha={alpha:<5} | OOF MSE={mse:.4f}")

        if mse < best["mse"]:
            best = {"alpha": float(alpha), "mse": float(mse)}
            best_oof = oof.copy()
            model.fit(train_text, y)
            best_test = model.predict(test_text)

    print("\nBest Text2:", best)
    return best_oof, best_test

oof_pred_text2, test_pred_text2 = fit_text2_oof(X_text, X_text_test, y_target)


alpha=1.0   | OOF MSE=156656.1284
alpha=2.0   | OOF MSE=155121.7532
alpha=5.0   | OOF MSE=159195.2586
alpha=10.0  | OOF MSE=167021.7665
alpha=20.0  | OOF MSE=178644.5315
alpha=50.0  | OOF MSE=198585.2167

Best Text2: {'alpha': 2.0, 'mse': 155121.75324291803}


### Base model #4: LightGBM with SVD text components

This block builds a hybrid base model that injects text information into a tree-based model. Listing titles are first converted into TF-IDF features, then compressed with `TruncatedSVD`, and the resulting dense components are appended to the tabular feature set. The final model is LightGBM trained on both tabular features and SVD-derived text features.

Binary string columns such as yes/no or true/false values are normalized to 0/1 when possible. This keeps them in the numeric branch of the preprocessing pipeline instead of expanding them into artificial one-hot categories.

The TF-IDF vocabulary and SVD projection are fitted on the combined train+test text as a competition-style transductive preprocessing step. This step does not use the target variable; it only defines a shared text representation for both datasets. The SVD components `svd_t_0 ... svd_t_119` are then added to both train and test matrices, while the raw text column is removed from the tabular model input.

Training follows the same OOF pattern as the previous models: preprocessing is fitted inside each fold, LightGBM uses early stopping, and the final model is trained on the full training data using the median best iteration. The resulting OOF and test predictions become another set of meta-features for the final stacker.


In [ ]:
def map_binary_series(s: pd.Series):
    if s.dtype != "object":
        return s
    ss = s.astype(str).str.strip().str.lower()
    ss = ss.replace({"nan": np.nan, "none": np.nan, "": np.nan})
    mapping = {"tak": 1, "nie": 0, "yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0}
    uniq = set(ss.dropna().unique().tolist())
    if uniq and uniq.issubset(set(mapping.keys())):
        return ss.map(mapping).astype("float")
    return s

def fit_lgb_with_svd_oof(train_df: pd.DataFrame, test_df: pd.DataFrame, y: np.ndarray, n_comp: int = 120):
    tr = train_df.copy()
    te = test_df.copy()

    for df in (tr, te):
        df.drop(columns=["price", "id", "quarter"], errors="ignore", inplace=True)

    for col in tr.columns:
        if col in [TEXT_COL, "quarter_clean"]:
            continue
        tr[col] = map_binary_series(tr[col])
        if col in te.columns:
            te[col] = map_binary_series(te[col])

    all_text = pd.concat([tr[TEXT_COL], te[TEXT_COL]], axis=0).fillna("").astype(str).values

    tfidf_union = FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2),
            min_df=3, max_df=0.9,
            strip_accents="unicode", lowercase=True, sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char", ngram_range=(3, 5),
            min_df=3, max_df=0.95,
            strip_accents="unicode", lowercase=True, sublinear_tf=True
        )),
    ])

    X_tfidf_all = tfidf_union.fit_transform(all_text)
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    X_svd_all = svd.fit_transform(X_tfidf_all)

    X_svd_tr = X_svd_all[:len(tr)]
    X_svd_te = X_svd_all[len(tr):]

    svd_cols = [f"svd_t_{i}" for i in range(n_comp)]
    tr = pd.concat([tr, pd.DataFrame(X_svd_tr, columns=svd_cols, index=tr.index)], axis=1)
    te = pd.concat([te, pd.DataFrame(X_svd_te, columns=svd_cols, index=te.index)], axis=1)

    X_tab_svd = tr.drop(columns=[TEXT_COL], errors="ignore")
    X_tab_svd_test = te.drop(columns=[TEXT_COL], errors="ignore")

    missing_in_test = sorted(set(X_tab_svd.columns) - set(X_tab_svd_test.columns))
    extra_in_test   = sorted(set(X_tab_svd_test.columns) - set(X_tab_svd.columns))
    assert not missing_in_test and not extra_in_test

    pre_svd = build_preprocessor(X_tab_svd)

    oof = np.zeros(len(X_tab_svd))
    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tab_svd)):
        X_tr, X_val = X_tab_svd.iloc[tr_idx], X_tab_svd.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        X_tr_t = pre_svd.fit_transform(X_tr)
        X_val_t = pre_svd.transform(X_val)

        lgbm = LGBMRegressor(
            n_estimators=20000,
            learning_rate=0.02,
            num_leaves=127,
            min_child_samples=25,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=2.0,
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1
        )
        lgbm.fit(
            X_tr_t, y_tr,
            eval_set=[(X_val_t, y_val)],
            eval_metric="l2",
            callbacks=[early_stopping(stopping_rounds=300, verbose=False)]
        )

        pred = lgbm.predict(X_val_t, num_iteration=lgbm.best_iteration_)
        oof[val_idx] = pred
        best_iters.append(lgbm.best_iteration_)
        print(f"Fold {fold+1} MSE: {mean_squared_error(y_val, pred):.4f} | best_iter={lgbm.best_iteration_}")

    best_n = int(np.median(best_iters))
    print("\nLGBM+SVD CV MSE:", mean_squared_error(y, oof))
    print("Best iters (median):", best_n)

    X_full_t = pre_svd.fit_transform(X_tab_svd)
    X_test_t = pre_svd.transform(X_tab_svd_test)

    lgbm_final = LGBMRegressor(
        n_estimators=best_n,
        learning_rate=0.02,
        num_leaves=127,
        min_child_samples=25,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=2.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    lgbm_final.fit(X_full_t, y)
    test_pred = lgbm_final.predict(X_test_t)

    return oof, test_pred

oof_pred_lgb_svd, test_pred_lgb_svd = fit_lgb_with_svd_oof(train_fe, test_fe, y_target, n_comp=120)
print("\nSaved preds: oof_pred_lgb_svd, test_pred_lgb_svd")


Fold 1 MSE: 76020.8658 | best_iter=1356
Fold 2 MSE: 83271.9566 | best_iter=1259
Fold 3 MSE: 84624.4858 | best_iter=1407
Fold 4 MSE: 83037.6302 | best_iter=1431
Fold 5 MSE: 86505.0032 | best_iter=2397

LGBM+SVD CV MSE: 82691.44913518315
Best iters (median): 1407

Saved preds: oof_pred_lgb_svd, test_pred_lgb_svd


### Final model: Polynomial Ridge stacking and submission export

This is the final stage of the project. Four base models are combined into one competition model: improved LightGBM tabular, XGBoost, LightGBM with SVD text components, and TF-IDF + Ridge text regression.

`X_meta` is a matrix with one column per base model. Each value in `X_meta` is an OOF prediction, meaning that the corresponding row was predicted by a model that did not see that row during training. This makes the stacking procedure methodologically correct. `X_meta_test` is built in the same way using predictions for the test set.

The meta-model is a `PolynomialFeatures(degree=2) + StandardScaler + Ridge` pipeline. A plain Ridge model would learn only linear weights for the base predictions. Adding second-degree polynomial features allows the meta-model to learn interactions between base models, for example cases where two models being high at the same time should change the final correction. Standardization is important because polynomial expansion creates features on different scales, and Ridge regularization helps reduce overfitting.

Several `alpha` values are evaluated using OOF MSE, and the best one is used to train the final meta-model on the full meta-feature matrix. Final predictions are clipped to the training target range as a simple guard against unrealistic extreme values. The submission file is saved to `outputs/submission_polystack.csv`, using the IDs from `sample_sub` to preserve the expected Kaggle format.


In [ ]:
X_meta = np.column_stack([
    oof_pred_tab2,
    oof_pred_xgb,
    oof_pred_lgb_svd,
    oof_pred_text2
])

X_meta_test = np.column_stack([
    test_pred_tab2,
    test_pred_xgb,
    test_pred_lgb_svd,
    test_pred_text2
])

alphas = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
best = {"alpha": None, "mse": np.inf}

for alpha in alphas:
    meta_poly = Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, random_state=RANDOM_STATE))
    ])

    oof = np.zeros(len(y_target))
    fold_mse = []

    for tr_idx, val_idx in kf.split(X_meta):
        meta_poly.fit(X_meta[tr_idx], y_target[tr_idx])
        pred = meta_poly.predict(X_meta[val_idx])
        oof[val_idx] = pred
        fold_mse.append(mean_squared_error(y_target[val_idx], pred))

    mse = mean_squared_error(y_target, oof)
    print(f"alpha={alpha:<4} | PolyStack OOF MSE={mse:.4f} | std={np.std(fold_mse):.2f}")

    if mse < best["mse"]:
        best = {"alpha": float(alpha), "mse": float(mse)}

print("\nBest PolyStack:", best)

meta_poly_final = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best["alpha"], random_state=RANDOM_STATE))
])
meta_poly_final.fit(X_meta, y_target)
test_pred_polystack = meta_poly_final.predict(X_meta_test)

min_price, max_price = float(y.min()), float(y.max())
test_pred_polystack = np.clip(test_pred_polystack, min_price, max_price)

sub = sample_sub.copy()
id_col = sub.columns[0]
target_col = sub.columns[1]
sub[target_col] = test_pred_polystack

out_path = OUTPUT_DIR / "submission_polystack.csv"
sub.to_csv(out_path, index=False)

print("\nSaved:", out_path)
print(sub.head().to_string(index=False))
print("ID check (first/last):", sub[id_col].iloc[0], sub[id_col].iloc[-1])


alpha=0.0  | PolyStack OOF MSE=76142.1847 | std=3178.16
alpha=0.1  | PolyStack OOF MSE=76103.0878 | std=3191.14
alpha=0.5  | PolyStack OOF MSE=76044.1498 | std=3192.52
alpha=1.0  | PolyStack OOF MSE=76003.2295 | std=3181.18
alpha=2.0  | PolyStack OOF MSE=75948.2073 | std=3159.97
alpha=5.0  | PolyStack OOF MSE=75861.5362 | std=3120.68
alpha=10.0 | PolyStack OOF MSE=75800.6740 | std=3090.82
alpha=20.0 | PolyStack OOF MSE=75757.3730 | std=3070.45

Best PolyStack: {'alpha': 20.0, 'mse': 75757.37299554014}

Saved: outputs/submission_polystack.csv
 ID  TARGET
  1 1935.03
  2 1216.07
  3 1240.99
  4 1893.34
  5 3720.77
ID check (first/last): 1 4842
